# Proyecto final: análisis de factores asociados a enfermedad cardíaca
**Estudiante:** Luis Miguel Reyes Prado  
**Modalidad:** Individual  
**Base:** Heart Failure Prediction Dataset (`heart.csv`)  

## Pregunta de análisis
**¿Qué variables clínicas y demográficas presentan las asociaciones descriptivas más claras con la presencia de enfermedad cardíaca en los 918 registros?**

> Alcance: análisis exploratorio. Las asociaciones observadas no demuestran causalidad ni sustituyen un diagnóstico clínico.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
DATA_PATH = Path("heart.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Revisión inicial
Se inspeccionan dimensiones, tipos, valores faltantes, duplicados y categorías. Los valores cero en `RestingBP` y `Cholesterol` se revisan aparte porque pueden ser códigos de ausencia y no mediciones clínicas plausibles.

In [ ]:
print("Dimensiones:", df.shape)
display(df.head(), df.tail())
display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().to_frame("nulos"))
print("Duplicados:", df.duplicated().sum())
for col in df.select_dtypes(include="object"):
    print(col, sorted(df[col].unique()))
print("Ceros RestingBP:", (df.RestingBP==0).sum())
print("Ceros Cholesterol:", (df.Cholesterol==0).sum())

## 2. Limpieza y transformación
- No hay nulos explícitos ni filas duplicadas.
- `RestingBP` contiene **1** cero. Se reemplaza por la mediana de valores positivos (**130 mmHg**), porque 0 mmHg no representa una medición viable y la mediana es robusta.
- `Cholesterol` contiene **172** ceros. Se interpretan como valores faltantes codificados. Se imputan con la mediana positiva del subgrupo de sexo y rango de edad, conservando diferencias básicas sin usar la variable objetivo.
- No se eliminan columnas: las 11 predictoras aportan contexto clínico y `HeartDisease` es el resultado.
- Se crea `AgeGroup` para comparaciones y `RiskProfile` como resumen descriptivo de angina de esfuerzo + pendiente ST.

In [ ]:
clean = df.copy()
clean.loc[clean.RestingBP==0, "RestingBP"] = clean.loc[clean.RestingBP>0, "RestingBP"].median()
clean["AgeGroup"] = pd.cut(clean.Age,[0,39,49,59,69,100],labels=["<40","40-49","50-59","60-69","70+"])
med = clean.loc[clean.Cholesterol>0].groupby(["Sex","AgeGroup"], observed=False).Cholesterol.median()
for i in clean.index[clean.Cholesterol==0]:
    v = med.get((clean.at[i,"Sex"], clean.at[i,"AgeGroup"]), np.nan)
    clean.at[i,"Cholesterol"] = v if pd.notna(v) else clean.loc[clean.Cholesterol>0,"Cholesterol"].median()
clean["RiskProfile"] = np.select([
    (clean.ExerciseAngina=="Y") & clean.ST_Slope.isin(["Flat","Down"]),
    (clean.ExerciseAngina=="Y") | clean.ST_Slope.isin(["Flat","Down"])
], ["Alto","Intermedio"], default="Bajo")
clean.to_csv("heart_limpio.csv", index=False)
clean.head()

## 3. Estadísticas descriptivas

In [ ]:
display(clean.describe(include="all").T)
display(clean.groupby("HeartDisease")[["Age","RestingBP","Cholesterol","MaxHR","Oldpeak"]].agg(["mean","median","std"]).round(2))
print((clean.HeartDisease.value_counts(normalize=True)*100).round(1))

## 4. Visualizaciones e interpretación

In [ ]:
plt.figure(figsize=(9,5)); sns.histplot(data=clean,x="Age",hue="HeartDisease",bins=12,kde=True,alpha=.55); plt.title("Distribución de edad según diagnóstico"); plt.xlabel("Edad (años)"); plt.ylabel("Pacientes"); plt.show()
# Interpretación: los registros con enfermedad se concentran relativamente más en edades mayores.

In [ ]:
rate=(clean.groupby("ChestPainType").HeartDisease.mean()*100).reindex(["TA","ATA","NAP","ASY"])
plt.figure(figsize=(9,5)); ax=sns.barplot(x=rate.index,y=rate.values,color="#457B9D"); ax.bar_label(ax.containers[0],fmt="%.1f%%"); plt.ylim(0,100); plt.title("Prevalencia por tipo de dolor torácico"); plt.xlabel("Tipo"); plt.ylabel("Con enfermedad (%)"); plt.show()
# Interpretación: ASY presenta la proporción más alta; la ausencia de dolor típico no equivale a bajo riesgo.

In [ ]:
plt.figure(figsize=(9,5)); sns.boxplot(data=clean,x="HeartDisease",y="MaxHR",hue="HeartDisease",legend=False); plt.xticks([0,1],["Sin enfermedad","Con enfermedad"]); plt.title("Frecuencia cardíaca máxima por diagnóstico"); plt.xlabel("Diagnóstico"); plt.ylabel("MaxHR (lpm)"); plt.show()
# Interpretación: el grupo con enfermedad tiende a alcanzar una frecuencia máxima menor.

In [ ]:
plt.figure(figsize=(9,5)); sns.scatterplot(data=clean,x="Age",y="Oldpeak",hue="HeartDisease",alpha=.7); plt.title("Edad y depresión ST según diagnóstico"); plt.xlabel("Edad"); plt.ylabel("Oldpeak"); plt.show()
# Interpretación: valores elevados de Oldpeak aparecen con mayor frecuencia en la clase con enfermedad.

In [ ]:
num=["Age","RestingBP","Cholesterol","FastingBS","MaxHR","Oldpeak","HeartDisease"]
plt.figure(figsize=(9,7)); sns.heatmap(clean[num].corr(),annot=True,fmt=".2f",cmap="RdBu_r",center=0,vmin=-1,vmax=1); plt.title("Correlaciones numéricas"); plt.show()
# Interpretación: Oldpeak se asocia positivamente y MaxHR negativamente con HeartDisease; las correlaciones no implican causalidad.

In [ ]:
rate=(clean.groupby("ST_Slope").HeartDisease.mean()*100).reindex(["Up","Flat","Down"])
plt.figure(figsize=(9,5)); ax=sns.barplot(x=rate.index,y=rate.values,color="#8D5A97"); ax.bar_label(ax.containers[0],fmt="%.1f%%"); plt.ylim(0,100); plt.title("Prevalencia según pendiente ST"); plt.xlabel("ST_Slope"); plt.ylabel("Con enfermedad (%)"); plt.show()
# Interpretación: Flat y Down concentran una prevalencia notablemente mayor que Up.

## 5. Conclusiones
1. La prevalencia observada es **55.3%** (508/918); la muestra no debe interpretarse como prevalencia poblacional.
2. La angina inducida por ejercicio diferencia fuertemente los grupos: **85.2%** con enfermedad cuando `Y`, frente a **35.1%** cuando `N`.
3. La pendiente ST es uno de los indicadores descriptivos más claros: `Flat` **82.8%**, `Down` **77.8%** y `Up` **19.7%**.
4. El dolor `ASY` alcanza **79.0%** de enfermedad, por lo que la categoría asintomática no debe leerse como ausencia de riesgo.
5. En mediana, el grupo con enfermedad tiene MaxHR de **126 lpm** frente a **150 lpm**, y Oldpeak de **1.2** frente a **0.0**.

### Limitaciones
Datos integrados de cinco fuentes históricas, ceros codificados en variables clínicas, análisis observacional, desbalance demográfico y ausencia de variables como medicación, tabaquismo o seguimiento temporal.